In [2]:
from langgraph.graph import START, StateGraph
from typing_extensions import TypedDict
from agents.deep import make_agent
tgraph = make_agent({})


In [3]:
tgraph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "你好",
            }
        ],
    },
    config={"configurable": {"thread_id": "9877"}},
)          

{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='786bbbee-3930-4175-abcc-4773957ef46d'),
  AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 1388, 'total_tokens': 1397, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen2.5-VL-32B-Instruct', 'system_fingerprint': None, 'id': 'chatcmpl-eb10b4df355f440fa64e399b9ada28e6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d50f3-b9e8-7971-b6e1-2e77b613d1d7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1388, 'output_tokens': 9, 'total_tokens': 1397, 'input_token_details': {}, 'output_token_details': {}})]}

In [3]:
import requests
import uuid
requests.delete(
    f"http://10.17.1.26:2024/assistants/677d3624-04b1-4797-9afb-a6d956533073"
)



<Response [204]>

In [1]:
import nest_asyncio
nest_asyncio.apply()

from langgraph_sdk import get_client
import asyncio

async def create_assistant(agent,name,configurable):
    client = get_client(url="http://localhost:2024")
    assistant = await client.assistants.create(
        graph_id="deep",
        name=name,
        config=configurable
    )
    return assistant
    


In [17]:
assistant1 = await create_assistant("agent","卡皮巴拉",
    {"configurable":{"system_prompt":"你是一只卡皮巴拉，请用很佛系的态度对待每一件事"}}
)
assistant1

{'assistant_id': '60304801-850a-4a66-86d4-eb6ceaa07b38',
 'graph_id': 'deep',
 'config': {'configurable': {'system_prompt': '你是一只卡皮巴拉，请用很佛系的态度对待每一件事'}},
 'context': {'system_prompt': '你是一只卡皮巴拉，请用很佛系的态度对待每一件事'},
 'metadata': {},
 'name': '卡皮巴拉',
 'created_at': '2026-04-03T02:51:48.413969+00:00',
 'updated_at': '2026-04-03T02:51:48.413969+00:00',
 'version': 1,
 'description': None}

In [18]:
assistant2 = await create_assistant("agent","狗",
    {"configurable":{"system_prompt":"你是一只狗，每次都用狗语回答"}}
)
assistant2

{'assistant_id': '4e08ba62-8212-4eef-926a-b3e6206d516c',
 'graph_id': 'deep',
 'config': {'configurable': {'system_prompt': '你是一只狗，每次都用狗语回答'}},
 'context': {'system_prompt': '你是一只狗，每次都用狗语回答'},
 'metadata': {},
 'name': '狗',
 'created_at': '2026-04-03T02:51:50.465795+00:00',
 'updated_at': '2026-04-03T02:51:50.465795+00:00',
 'version': 1,
 'description': None}

In [28]:
# 流式调用版本
async def call_deep_agent_stream(assistant_id,thread=None):
    client = get_client(url="http://localhost:2024")    
    if not thread:
        thread = await client.threads.create()
    async for chunk in client.runs.stream(
        thread_id=thread["thread_id"],
        assistant_id=assistant_id,
        input={
            "messages": [{"role": "user", "content": "你是谁？"}]
        },
        stream_mode="messages"  # 或 "values", "updates"
    ):
        if chunk.data and isinstance(chunk.data, list) and (chunk.data)[0]['content']:
            print((chunk.data)[0]['type'],":",(chunk.data)[0]['content'])


await call_deep_agent_stream(assistant1['assistant_id'])


ai : 我
ai : 我是一只
ai : 我是一只卡
ai : 我是一只卡皮
ai : 我是一只卡皮巴拉
ai : 我是一只卡皮巴拉，
ai : 我是一只卡皮巴拉，用
ai : 我是一只卡皮巴拉，用很
ai : 我是一只卡皮巴拉，用很佛系
ai : 我是一只卡皮巴拉，用很佛系的态度
ai : 我是一只卡皮巴拉，用很佛系的态度对待
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。


ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手，
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手，通过
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手，通过工具
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手，通过工具帮助
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手，通过工具帮助用户
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手，通过工具帮助用户完成任务
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手，通过工具帮助用户完成任务。
ai : 我是一只卡皮巴拉，用很佛系的态度对待每一件事。

我是Deep Agent，一个AI助手，通过工具帮助用户完成任务。
ai : 我是